In [ ]:
cd ..

In [ ]:
# Import necessary libraries
import nltk
nltk.download('punkt')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score
from rouge_score import rouge_scorer
from bert_score import score as bert_score_func
import torch

# For BARTScore and GPTScore
from transformers import BartForConditionalGeneration, BartTokenizer
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# For QSTS and QRelScore
from sentence_transformers import SentenceTransformer, util

# # Define the context sentence and generated MCQ question
# context_sentence = "Machine learning is a field of artificial intelligence that uses algorithms to learn from and make predictions on data."
# generated_mcq_question = "What is machine learning?"
# options = [
#     "A field of artificial intelligence that uses algorithms",
#     "A form of supervised learning only",
#     "A programming language",
#     "A method of organizing data"
# ]
# # context_sentence = "A high-degree polynomial kernel introduces high complexity to the decision boundary, which can lead to overfitting, especially when the training dataset is small. The SVM model may fit the noise in the training data rather than capturing the underlying pattern, reducing generalization to unseen data."
# # generated_mcq_question = "In the context of Support Vector Machines (SVM), which of the following scenarios is most likely to lead to overfitting when classifying a dataset?"
# # options = [
# #     "Choosing a linear kernel for a dataset that is linearly separable.",
# #     "Using a high-degree polynomial kernel on a small training dataset.",
# #     "Selecting a Gaussian (RBF) kernel with a large value for the hyperparameter 𝛾"
# #     "Setting the regularization parameter C to a very small value."
# # ]

# mcq_text = generated_mcq_question + " " + " ".join(options)

In [ ]:
import os
import sys

import gradio as gr

from dotenv import load_dotenv
from core.retriever.Retriever import Retriever
from core.ingestion.preprocessing.storage.FaissStore import FaissStore
from core.llm.TeacherLLM import TeacherBot
from core.llm.AssistantLLM import AssistantBot

# Load environment variables from a .env file
load_dotenv()
# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

In [ ]:
def compute_bleu4(context: str = None, mcq_text: str = None) -> float:
    """
    Compute BLEU-4 score between the context and generated MCQ question.
    """
    # Tokenize the context and generated MCQ question
    reference = nltk.word_tokenize(context.lower())
    candidate = nltk.word_tokenize(mcq_text.lower())
    
    # Calculate BLEU-4 score
    bleu_score = sentence_bleu([reference], candidate, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=SmoothingFunction().method1)
    
    return bleu_score

In [ ]:
print(f"BLEU-4 score: {compute_bleu4(context_sentence, mcq_text):.4f}")

In [ ]:
def compute_rougeL(context: str = None, mcq_text: str = None) -> float:
    """
    Compute ROUGE-L score between the context and generated MCQ question.
    """
    # Initialize ROUGE scorer
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    
    # Calculate ROUGE-L score
    scores = scorer.score(context, mcq_text)
    
    return scores['rougeL'].fmeasure

In [ ]:
print(f"ROUGE-L Score: {compute_rougeL(context_sentence, mcq_text):.4f}")

In [ ]:
def compute_qsts(context: str = None, mcq_text: str = None):
    """
    Compute QSTS score between the context and generated MCQ question.
    """
    # Load the QSTS model
    qsts_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    
    # Encode the context and generated MCQ question
    context_embedding = qsts_model.encode(context, convert_to_tensor=True)
    mcq_text_embedding = qsts_model.encode(mcq_text, convert_to_tensor=True)
    
    # Calculate cosine similarity
    cosine_similarity = util.pytorch_cos_sim(context_embedding, mcq_text_embedding)
    
    return cosine_similarity.item()

In [ ]:
print(f"QSTS (Semantic Similarity): {compute_qsts(context_sentence, mcq_text):.4f}")